# Generative Adversarial Networks (GANs)

**Two neural networks playing a game:**
- **Generator**: Creates fake data to fool the discriminator
- **Discriminator**: Tries to distinguish real from fake

Topics:
- GAN theory and training dynamics
- **Simple GAN** on MNIST
- **DCGAN** (Deep Convolutional GAN)
- Training tips and common failures

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# Load MNIST
(X_train, y_train), (_, _) = mnist.load_data()
X_train = X_train.astype('float32') / 255.0
X_train = X_train.reshape(-1, 784)  # Flatten for simple GAN
print(f'Training data: {X_train.shape}')

## 1. GAN Architecture

```
Random Noise (z) ─→ [Generator] ─→ Fake Image ─┐
                                                  ├─→ [Discriminator] ─→ Real/Fake?
Real Image ───────────────────────────────────────┘
```

**Training alternates:**
1. Train Discriminator: maximize `log(D(real)) + log(1 - D(fake))`
2. Train Generator: maximize `log(D(fake))` (fool the discriminator)

In [ ]:
# Hyperparameters
latent_dim = 100
img_dim = 784

# Generator
def build_generator():
    model = keras.Sequential([
        layers.Dense(256, input_dim=latent_dim),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(512),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(1024),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(momentum=0.8),
        layers.Dense(img_dim, activation='sigmoid')
    ], name='generator')
    return model

# Discriminator
def build_discriminator():
    model = keras.Sequential([
        layers.Dense(512, input_dim=img_dim),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Dense(256),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ], name='discriminator')
    return model

generator = build_generator()
discriminator = build_discriminator()

# Compile discriminator
discriminator.compile(loss='binary_crossentropy', optimizer=keras.optimizers.Adam(0.0002, 0.5), metrics=['accuracy'])

# GAN (generator + frozen discriminator)
discriminator.trainable = False
gan_input = keras.Input(shape=(latent_dim,))
generated_img = generator(gan_input)
validity = discriminator(generated_img)
gan = keras.Model(gan_input, validity)
gan.compile(loss='binary_crossentropy', optimizer=keras.optimizers.Adam(0.0002, 0.5))

print(f'Generator params: {generator.count_params():,}')
print(f'Discriminator params: {discriminator.count_params():,}')

In [ ]:
def train_gan(epochs=5000, batch_size=64, sample_interval=1000):
    """Train GAN and show generated samples."""
    real_label = np.ones((batch_size, 1))
    fake_label = np.zeros((batch_size, 1))
    
    d_losses, g_losses = [], []
    
    for epoch in range(epochs):
        # --- Train Discriminator ---
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        real_imgs = X_train[idx]
        
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_imgs = generator.predict(noise, verbose=0)
        
        d_loss_real = discriminator.train_on_batch(real_imgs, real_label)
        d_loss_fake = discriminator.train_on_batch(fake_imgs, fake_label)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        # --- Train Generator ---
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, real_label)  # We want D to think these are real
        
        d_losses.append(d_loss[0])
        g_losses.append(g_loss)
        
        if (epoch + 1) % sample_interval == 0:
            print(f'Epoch {epoch+1}/{epochs} | D Loss: {d_loss[0]:.4f} | D Acc: {d_loss[1]*100:.1f}% | G Loss: {g_loss:.4f}')
            show_generated(epoch + 1)
    
    return d_losses, g_losses

def show_generated(epoch, n=10):
    noise = np.random.normal(0, 1, (n, latent_dim))
    generated = generator.predict(noise, verbose=0).reshape(-1, 28, 28)
    fig, axes = plt.subplots(1, n, figsize=(14, 1.5))
    for i, ax in enumerate(axes):
        ax.imshow(generated[i], cmap='gray'); ax.axis('off')
    plt.suptitle(f'Generated Digits — Epoch {epoch}', fontsize=11)
    plt.tight_layout()
    plt.show()

d_losses, g_losses = train_gan(epochs=5000, batch_size=64, sample_interval=1000)

In [ ]:
# Training dynamics
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d_losses, alpha=0.5, label='Discriminator Loss')
ax.plot(g_losses, alpha=0.5, label='Generator Loss')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('GAN Training Dynamics')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. DCGAN — Convolutional GAN

In [ ]:
# DCGAN Generator
def build_dcgan_generator():
    model = keras.Sequential([
        layers.Dense(7 * 7 * 128, input_dim=latent_dim),
        layers.Reshape((7, 7, 128)),
        layers.BatchNormalization(),
        layers.Conv2DTranspose(64, 4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.BatchNormalization(),
        layers.Conv2DTranspose(1, 4, strides=2, padding='same', activation='sigmoid'),
    ], name='dcgan_generator')
    return model

# DCGAN Discriminator
def build_dcgan_discriminator():
    model = keras.Sequential([
        layers.Conv2D(64, 4, strides=2, padding='same', input_shape=(28, 28, 1)),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, 4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ], name='dcgan_discriminator')
    return model

dc_gen = build_dcgan_generator()
dc_disc = build_dcgan_discriminator()
print(f'DCGAN Generator: {dc_gen.count_params():,} params')
print(f'DCGAN Discriminator: {dc_disc.count_params():,} params')
dc_gen.summary()

## 3. GAN Training Tips

| Tip | Why |
|-----|-----|
| Use **LeakyReLU** (slope 0.2) | Prevents dying neurons |
| **BatchNorm** in generator | Stabilizes training |
| **Dropout** in discriminator | Prevents discriminator from being too strong |
| **Adam** with lr=0.0002, β₁=0.5 | Standard GAN optimizer settings |
| **Label smoothing** (0.9 instead of 1.0) | Prevents discriminator overconfidence |
| Add **noise to labels** | Regularizes discriminator |

### Common Failure Modes:
1. **Mode Collapse**: Generator only produces one type of output → Use minibatch discrimination
2. **Vanishing Gradients**: Discriminator too strong → Use Wasserstein loss (WGAN)
3. **Training Instability**: Oscillating losses → Use spectral normalization

### GAN Variants:
- **WGAN**: Wasserstein distance, more stable training
- **Conditional GAN (cGAN)**: Generate specific classes
- **StyleGAN**: State-of-the-art face generation
- **CycleGAN**: Image-to-image translation (horse → zebra)
- **Pix2Pix**: Paired image translation